# 🤖 Agents & Orchestration: Zero to Hero — A Guided Lab

An **agent** is an LLM that can *act*: it reasons about a goal, chooses **tools**, observes the
results, and loops until done. This lab builds agents from scratch — the ReAct loop, tool use,
memory, safety approval gates, and multi-agent orchestration.

**Runs 100% offline** with a rule-based mock "reasoner" standing in for the LLM, so you see the
control flow clearly. The loop is identical to real agent frameworks.

**Prerequisite:** the LangChain lab (tools & the agent idea).

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Agent vs chain (why agents?)
2. Tools with schemas
3. The ReAct loop (Reason → Act → Observe)
4. Building a single-tool agent
5. Multi-tool agents & tool selection
6. Agent memory & scratchpad
7. Stopping conditions & loop safety
8. Human-in-the-loop approval gates
9. Multi-agent orchestration
10. 🏆 Capstone: an orchestrated research assistant


In [ ]:
import re, json, math

# A mock "reasoner" that plays the role of the LLM's decision-making.
# In production this is an LLM prompted to output the next action as JSON.
class MockReasoner:
    def decide(self, goal, tools, scratchpad):
        g = goal.lower()
        used = [step["tool"] for step in scratchpad]
        # decide next action based on the goal and what's already been done
        if re.search(r"\d+\s*[+\-*/]\s*\d+", goal) and "calculator" not in used:
            expr = re.search(r"[-0-9+*/(). ]+", goal.split("calculate")[-1] if "calculate" in g else goal)
            return {"action": "calculator", "input": expr.group().strip() if expr else goal}
        if ("length" in g or "how many characters" in g) and "string_length" not in used:
            return {"action": "string_length", "input": goal.split(":")[-1].strip()}
        if ("weather" in g) and "get_weather" not in used:
            city = goal.split("in")[-1].strip(" ?.")
            return {"action": "get_weather", "input": city}
        # nothing left to do -> finish
        return {"action": "finish", "input": ""}

reasoner = MockReasoner()
print("Mock reasoner ready.")

---
## Chapter 1 — Agent vs Chain (why agents?)

📖 **Theory.** A **chain** runs a fixed sequence of steps. An **agent** decides its steps
*dynamically*: it looks at the goal and current state, picks the next action, and repeats. Use an
agent when the path isn't known in advance (variable number of steps, tool choice depends on
intermediate results).

🖼️ **Diagram — fixed vs dynamic control flow**
```
 CHAIN (fixed):    A ─► B ─► C ─► done

 AGENT (dynamic):  ┌─► reason ─► act ─► observe ─┐
                   └───────── loop until done ◄──┘
```

🧠 **Mental model.** A chain is a *railway* (one track). An agent is a *driver* (chooses turns at
each intersection based on what it sees).


In [ ]:
# Chains are predictable; agents are flexible. Trade-off: control vs. capability.
comparison = {
    "chain":  {"path": "fixed", "predictable": True,  "good_for": "known workflows"},
    "agent":  {"path": "dynamic", "predictable": False, "good_for": "open-ended tasks"},
}
for k, v in comparison.items():
    print(k, "->", v)

### ✏️ Your Turn 1.1
In a comment, give one task best solved by a **chain** and one best solved by an **agent**, and
say why.

In [ ]:
# chain task: ...
# agent task: ...


✅ **Solution**
```python
# chain: "translate then summarize" — fixed 2-step path, always the same.
# agent: "answer this question, using web search or a calculator as needed" —
#        the tools required depend on the question, decided at runtime.
```

---
## Chapter 2 — Tools with Schemas

📖 **Theory.** An agent's tools each need: a **name**, a **description** (when to use it), and a
clear **input** contract. The description is critical — it's how the LLM decides which tool fits.

🖼️ **Diagram — a tool the agent can reason about**
```
 name:        "get_weather"
 description: "get current weather for a city; input: city name"
 func:        city -> "Sunny, 24°C"
```


In [ ]:
class Tool:
    def __init__(self, name, description, func):
        self.name, self.description, self.func = name, description, func
    def run(self, arg):
        try: return str(self.func(arg))
        except Exception as e: return f"tool error: {e}"

def safe_calc(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr): return "invalid expression"
    return str(eval(expr))

def fake_weather(city):
    table = {"paris": "Sunny, 24C", "london": "Rainy, 15C", "tokyo": "Cloudy, 20C"}
    return table.get(city.lower().strip(), "Weather data unavailable")

tools = {
    "calculator":    Tool("calculator", "evaluate arithmetic; input: expression", safe_calc),
    "string_length": Tool("string_length", "count characters; input: text", lambda s: len(s)),
    "get_weather":   Tool("get_weather", "current weather; input: city", fake_weather),
}
for name, t in tools.items():
    print(f"{name}: {t.description}")

### ✏️ Your Turn 2.1
Add a `reverse_string` tool (description + func) to the `tools` dict and run it on `"agent"`.

In [ ]:
# add tools["reverse_string"] and test


✅ **Solution**
```python
tools["reverse_string"] = Tool("reverse_string", "reverse text; input: text", lambda s: s[::-1])
print(tools["reverse_string"].run("agent"))   # "tnega"
```

---
## Chapter 3 — The ReAct Loop (Reason → Act → Observe)

📖 **Theory.** The core agent algorithm is **ReAct**: the LLM alternates **Reasoning** ("I should
use the calculator") and **Acting** (calling the tool), feeding each **Observation** back into the
next reasoning step. It loops until it decides to **finish**.

🖼️ **Diagram — the ReAct cycle**
```
        ┌──────────────────────────────────────────┐
   goal │  1 REASON: which tool & input?           │
    ──► │  2 ACT: run the tool                      │
        │  3 OBSERVE: record the result             │
        │  4 goto 1  (until action == "finish")     │
        └──────────────────────────────────────────┘
```

🧠 **Mental model.** Think of the agent keeping a **scratchpad**: "I did X, saw Y, so next I'll
do Z." That running log is what makes multi-step reasoning possible.


In [ ]:
def react_loop(goal, tools, reasoner, max_steps=5, verbose=True):
    scratchpad = []       # list of {tool, input, observation}
    for step in range(max_steps):
        decision = reasoner.decide(goal, tools, scratchpad)
        action, arg = decision["action"], decision["input"]
        if action == "finish":
            if verbose: print(f"  step {step}: FINISH")
            break
        observation = tools[action].run(arg) if action in tools else f"unknown tool {action}"
        if verbose:
            print(f"  step {step}: REASON->{action}('{arg}')  OBSERVE->{observation}")
        scratchpad.append({"tool": action, "input": arg, "observation": observation})
    return scratchpad

print("Goal: calculate 15 * 8")
pad = react_loop("calculate 15 * 8", tools, reasoner)
print("final scratchpad:", pad)

⚠️ **Common trap.** Without a `max_steps` cap (and a real "finish" signal), an agent can loop
forever — repeating the same tool or oscillating. Always bound the loop (Ch.7).

### ✏️ Your Turn 3.1
Run the loop for the goal `"what is the weather in Tokyo?"` and confirm it calls `get_weather`
and finishes.

In [ ]:
# react_loop("what is the weather in Tokyo?", tools, reasoner)


✅ **Solution**
```python
pad = react_loop("what is the weather in Tokyo?", tools, reasoner)
# step 0: get_weather('Tokyo') -> Cloudy, 20C ; step 1: FINISH
```

---
## Chapter 4 — Building a Single-Tool Agent

📖 **Theory.** Wrap the ReAct loop into an `Agent` class that produces a **final answer** by
summarizing its scratchpad. This is the smallest complete agent.


In [ ]:
class Agent:
    def __init__(self, tools, reasoner, max_steps=5):
        self.tools, self.reasoner, self.max_steps = tools, reasoner, max_steps
    def run(self, goal, verbose=False):
        scratchpad = react_loop(goal, self.tools, self.reasoner, self.max_steps, verbose)
        if not scratchpad:
            return "I could not find a tool to help with that."
        # final answer = the last observation (a real agent asks the LLM to summarize)
        last = scratchpad[-1]
        return f"Result: {last['observation']} (via {last['tool']})"

agent = Agent(tools, reasoner)
print(agent.run("calculate 42 * 7"))
print(agent.run("weather in London"))

### ✏️ Your Turn 4.1
Use the agent to compute the length of the string after the colon in
`"string length of this: orchestration"`. What does it return?

In [ ]:
# agent.run("string length of this: orchestration")


✅ **Solution**
```python
print(agent.run("string length of this: orchestration"))
# Result: 13 (via string_length)
```

---
## Chapter 5 — Multi-Tool Agents & Tool Selection

📖 **Theory.** With several tools, the agent must **select** the right one each step. Real agents
let the LLM choose based on tool **descriptions**; here our reasoner routes by keywords. The key
skill is writing tools whose descriptions make selection unambiguous.

🖼️ **Diagram — selection among tools**
```
 goal ─►[ reasoner reads all tool descriptions ]─► picks best-matching tool ─► act
```


In [ ]:
# A goal needing multiple different tools across steps
class MultiStepReasoner(MockReasoner):
    def decide(self, goal, tools, scratchpad):
        used = [s["tool"] for s in scratchpad]
        g = goal.lower()
        # a compound goal: weather AND a calculation
        if "weather" in g and "get_weather" not in used:
            city = re.search(r"weather in (\w+)", g)
            return {"action":"get_weather","input": city.group(1) if city else "paris"}
        if re.search(r"\d+\s*[+\-*/]\s*\d+", goal) and "calculator" not in used:
            expr = re.search(r"\d+\s*[+\-*/]\s*\d+", goal).group()
            return {"action":"calculator","input": expr}
        return {"action":"finish","input":""}

multi = MultiStepReasoner()
agent2 = Agent(tools, multi)
print("Compound goal: 'weather in paris and compute 3 * 9'")
pad = react_loop("weather in paris and compute 3 * 9", tools, multi)
print("used tools:", [s["tool"] for s in pad])

⚡ **Pro tip.** If an agent keeps picking the wrong tool, the fix is almost always **better
tool descriptions**, not a smarter model. Make each description state exactly *when* to use it and
*what input* it needs.

### ✏️ Your Turn 5.1
State (in a comment) how you'd rewrite the `get_weather` description so an agent never confuses it
with a hypothetical `get_forecast` (multi-day) tool.

In [ ]:
# improved description: ...


✅ **Solution**
```python
# "get_weather: CURRENT weather for ONE city right now (not forecasts). Input: a single city name."
# The words CURRENT and 'not forecasts' disambiguate it from a multi-day forecast tool.
```

---
## Chapter 6 — Agent Memory & Scratchpad

📖 **Theory.** Two kinds of memory:
- **Scratchpad (working memory):** the current task's step log — reasoning + observations.
- **Long-term memory:** facts carried *across* tasks (past results, user preferences).

The scratchpad is what lets step N use the result of step N-1.

🖼️ **Diagram — scratchpad feeds reasoning**
```
 scratchpad: [ (calc, 15*8, 120) ]
                     │ fed back in
                     ▼
 reason: "I already computed 120, now I can compare it..."
```


In [ ]:
class MemoryAgent(Agent):
    def __init__(self, tools, reasoner, max_steps=5):
        super().__init__(tools, reasoner, max_steps)
        self.long_term = []          # persists across .run() calls
    def run(self, goal, verbose=False):
        result = super().run(goal, verbose)
        self.long_term.append({"goal": goal, "result": result})
        return result

mem_agent = MemoryAgent(tools, reasoner)
mem_agent.run("calculate 10 * 10")
mem_agent.run("weather in Paris")
print("long-term memory (across tasks):")
for m in mem_agent.long_term:
    print(" ", m)

### ✏️ Your Turn 6.1
Add a method `recall(keyword)` to `MemoryAgent` that returns past tasks whose goal contains the
keyword. Test with `recall("weather")`.

In [ ]:
# add recall(keyword) and test


✅ **Solution**
```python
def recall(self, keyword):
    return [m for m in self.long_term if keyword.lower() in m["goal"].lower()]
MemoryAgent.recall = recall
print(mem_agent.recall("weather"))
```

---
## Chapter 7 — Stopping Conditions & Loop Safety

📖 **Theory.** Agents must **terminate**. Safety mechanisms:
- **max_steps** — hard cap on iterations.
- **repeat detection** — stop if the same (tool, input) repeats (a loop).
- **no-progress** — stop if nothing new is learned.

🖼️ **Diagram — guarded loop**
```
 each step:  step<max? ──no─► STOP
             repeated action? ──yes─► STOP
             else ─► act
```


In [ ]:
def safe_react_loop(goal, tools, reasoner, max_steps=5):
    scratchpad, seen = [], set()
    for step in range(max_steps):
        decision = reasoner.decide(goal, tools, scratchpad)
        action, arg = decision["action"], decision["input"]
        if action == "finish":
            return scratchpad, "finished normally"
        key = (action, arg)
        if key in seen:                       # repeat detection
            return scratchpad, f"stopped: repeated action {key}"
        seen.add(key)
        obs = tools[action].run(arg) if action in tools else "unknown tool"
        scratchpad.append({"tool": action, "input": arg, "observation": obs})
    return scratchpad, "stopped: hit max_steps"

# a reasoner that would loop forever without the guard
class LoopyReasoner:
    def decide(self, goal, tools, scratchpad):
        return {"action": "calculator", "input": "1 + 1"}   # always the same!

pad, reason = safe_react_loop("loop test", tools, LoopyReasoner(), max_steps=10)
print("stop reason:", reason, "| steps taken:", len(pad))

⚠️ **Common trap.** A confidently-wrong agent can burn your entire API budget looping. In
production, always set BOTH a step cap and a cost/time budget, and log every step for debugging.

### ✏️ Your Turn 7.1
Modify `safe_react_loop` to also stop if the scratchpad reaches a `budget` number of *successful*
tool calls (e.g. budget=3), even if max_steps is higher.

In [ ]:
# add a budget parameter that limits successful tool calls


✅ **Solution**
```python
# inside the loop, after appending to scratchpad:
# if len(scratchpad) >= budget:
#     return scratchpad, "stopped: hit tool-call budget"
```

---
## Chapter 8 — Human-in-the-Loop Approval Gates

📖 **Theory.** Some actions are **irreversible or sensitive** (send email, spend money, delete
data). A safe agent **pauses for human approval** before those. Mark tools as requiring approval;
the loop requests confirmation before running them.

🖼️ **Diagram — the approval gate**
```
 agent wants to run a tool
        │
   sensitive? ──no──► run immediately
        │
       yes ──► ask human ──approve?──► run   /  ──deny──► skip + record
```


In [ ]:
class GatedTool(Tool):
    def __init__(self, name, description, func, requires_approval=False):
        super().__init__(name, description, func)
        self.requires_approval = requires_approval

def send_email(to):  # pretend this really sends
    return f"email sent to {to}"

gated_tools = dict(tools)
gated_tools["send_email"] = GatedTool("send_email", "send an email; input: recipient",
                                      send_email, requires_approval=True)

def run_with_approval(tool, arg, approve_fn):
    if getattr(tool, "requires_approval", False):
        if not approve_fn(tool.name, arg):
            return f"[DENIED] {tool.name}({arg}) was not approved"
    return tool.run(arg)

# simulate a human who approves calculators but not emails
def human(tool_name, arg):
    decision = tool_name != "send_email"
    print(f"  APPROVAL REQUEST: {tool_name}('{arg}') -> {'approved' if decision else 'DENIED'}")
    return decision

print(run_with_approval(gated_tools["send_email"], "boss@company.com", human))
print(run_with_approval(gated_tools["calculator"], "2 + 2", human))

⚡ **Pro tip.** Default to **deny** for sensitive actions when no human responds. "Fail
closed" (do nothing) is far safer than "fail open" (act without approval).

### ✏️ Your Turn 8.1
Write an `auto_approver(allowed_tools)` that approves only tools whose names are in a whitelist
set, and use it with `run_with_approval`.

In [ ]:
def auto_approver(allowed_tools):
    pass


✅ **Solution**
```python
def auto_approver(allowed_tools):
    return lambda name, arg: name in allowed_tools
approve = auto_approver({"calculator", "get_weather"})
print(run_with_approval(gated_tools["send_email"], "x@y.com", approve))  # DENIED
```

---
## Chapter 9 — Multi-Agent Orchestration

📖 **Theory.** Complex goals are split across **specialized agents** coordinated by an
**orchestrator**. Common pattern: a **planner** breaks the goal into subtasks, **worker** agents
solve each, and the orchestrator combines results.

🖼️ **Diagram — orchestrator + workers**
```
                ┌──────────────┐
       goal ──► │ ORCHESTRATOR │
                └──────┬───────┘
          ┌───────────┼───────────┐
          ▼           ▼           ▼
      [ math ]    [ weather ]  [ text ]     ← specialized worker agents
          └───────────┼───────────┘
                       ▼
                 combined answer
```

🧠 **Mental model.** Think of a manager delegating to specialists, then assembling their reports.
Each agent stays simple; the orchestrator handles coordination.


In [ ]:
# Specialized single-purpose agents
math_agent    = Agent({"calculator": tools["calculator"]}, reasoner)
weather_agent = Agent({"get_weather": tools["get_weather"]}, reasoner)

class Orchestrator:
    def __init__(self, workers): self.workers = workers  # dict name->agent
    def route(self, subtask):
        s = subtask.lower()
        if "weather" in s: return "weather", self.workers["weather"].run(subtask)
        if re.search(r"\d+\s*[+\-*/]\s*\d+", subtask): return "math", self.workers["math"].run(subtask)
        return "none", "no suitable agent"
    def solve(self, subtasks):
        return [{"subtask": st, "worker": self.route(st)[0], "result": self.route(st)[1]}
                for st in subtasks]

orch = Orchestrator({"math": math_agent, "weather": weather_agent})
plan = ["weather in Paris", "compute 25 * 4"]     # a planner would generate these
for r in orch.solve(plan):
    print(r)

⚠️ **Common trap.** Multi-agent systems multiply cost and latency (each agent may call the LLM
several times) and can cascade errors. Only reach for multi-agent when a single agent genuinely
can't cope — simpler is usually better.

### ✏️ Your Turn 9.1
Add a `text_agent` (using `string_length`) to the orchestrator and route a subtask like
`"length of: orchestration"` to it.

In [ ]:
# add text_agent and route a length subtask


✅ **Solution**
```python
text_agent = Agent({"string_length": tools["string_length"]}, reasoner)
orch.workers["text"] = text_agent
# extend route(): if "length" in s: return "text", self.workers["text"].run(subtask)
```

---
## 🏆 Chapter 10 — Capstone: Orchestrated Research Assistant

Build a `ResearchAssistant` that, given a compound goal, **plans** subtasks, dispatches them to
the right worker agent, enforces an **approval gate** on any sensitive tool, respects a **step
budget**, and returns a combined report. Build it before revealing the solution.

In [ ]:
# Your ResearchAssistant here
class ResearchAssistant:
    def __init__(self, tools, reasoner):
        pass
    def plan(self, goal):
        pass
    def run(self, goal):
        pass

# ra = ResearchAssistant(tools, reasoner)
# print(ra.run("weather in Tokyo and compute 12 * 12"))


✅ **Capstone Solution**
```python
class ResearchAssistant:
    def __init__(self, tools, reasoner):
        self.math = Agent({"calculator": tools["calculator"]}, reasoner)
        self.weather = Agent({"get_weather": tools["get_weather"]}, reasoner)
        self.text = Agent({"string_length": tools["string_length"]}, reasoner)
    def plan(self, goal):
        # naive planner: split on "and"; a real one asks the LLM for a task list
        return [p.strip() for p in re.split(r"\band\b", goal) if p.strip()]
    def _route(self, subtask):
        s = subtask.lower()
        if "weather" in s: return "weather", self.weather.run(subtask)
        if re.search(r"\d+\s*[+\-*/]\s*\d+", subtask): return "math", self.math.run(subtask)
        if "length" in s: return "text", self.text.run(subtask)
        return "none", "no suitable worker"
    def run(self, goal):
        report = []
        for st in self.plan(goal):
            worker, result = self._route(st)
            report.append({"subtask": st, "worker": worker, "result": result})
        return report

ra = ResearchAssistant(tools, reasoner)
for row in ra.run("weather in Tokyo and compute 12 * 12 and length of: agents"):
    print(row)
```

🎉 **You can build agents!** You understand the ReAct loop, tool selection, working vs long-term
memory, loop safety, human approval gates, and multi-agent orchestration. Swap the mock reasoner
for a real LLM (prompted to emit the next action as JSON) and these exact patterns run in
production.

---
### 📌 Concept Quick-Reference
**Agent vs chain:** chain = fixed path; agent = decides path at runtime
**Tools:** name + description + input contract; descriptions drive selection
**ReAct loop:** Reason -> Act -> Observe -> repeat until finish
**Memory:** scratchpad (this task) vs long-term (across tasks)
**Safety:** max_steps, repeat detection, cost/time budget, fail-closed
**Approval gates:** pause for human on irreversible/sensitive actions
**Orchestration:** planner -> worker agents -> combine (only when needed)
